# 01 — Foundation: docs → chunks → retrieval + NGO lookup

Runs end-to-end in `ds-general` AND Colab (free tier, offline default).
Colab first cell: `%pip install -q pandas scikit-learn`.
Uses `data/processed/_SAMPLE_DO_NOT_CITE.txt` until real legal docs land (see `data/raw/SOURCES.md`).

In [ ]:
import sys; sys.path.insert(0, '../src')
from load import load_txt
from chunk import recursive_split, section_aware_split
from retrieve import TfidfRetriever
from ngo import load_ngo, find_ngo, HELPLINES

In [ ]:
text = load_txt('../data/processed/_SAMPLE_DO_NOT_CITE.txt')
print(f'chars: {len(text)}')
print(text[:300])

In [ ]:
base = recursive_split(text, size=500, overlap=50)
sec = section_aware_split(text, size=800)
print(f'baseline chunks: {len(base)} (avg {sum(map(len, base)) / len(base):.0f} chars)')
print(f'section-aware chunks: {len(sec)} (avg {sum(map(len, sec)) / len(sec):.0f} chars)')

In [ ]:
questions = [
    'What does the Act say about discrimination?',
    'Are public buildings required to be accessible?',
    'What employment protections exist for persons with disabilities?',
    'Is there a commission for persons with disabilities?',
    'What are the penalties for violating the Act?',
    'Does the Act cover access to vehicles and transport?',
    'What rights do children with disabilities have to education?',
    'How long is the transitional compliance period?',
    'Does the Constitution guarantee dignity and equality?',
    'Where can I get legal aid for a disability rights violation?',
]
ret = TfidfRetriever(section_aware_split(text))
for q in questions:
    i, s, c = ret.query(q, k=1)[0]
    print(f'{s:.3f} | {q}  -> chunk {i}: {c[:80]}...')

In [ ]:
print('HELPLINES (always on top in UI):')
for h in HELPLINES: print(f" - {h['name']}: {h['phone']}")
df = load_ngo('../data/ngo.csv')
print(f'\nNGO rows: {len(df)} (spec minimum: 10)')
print(find_ngo(df, disability='visual', location='Lagos')[['name', 'phone']].to_string(index=False))
print(find_ngo(df, disability='hearing', location='Abuja')[['name', 'phone']].to_string(index=False))

## Stop gate
Foundation is done when this notebook runs top-to-bottom with ≥10 NGO rows and retrieval scores above.
Next (out of scope here): replace SAMPLE with real Act text, wire Gemini LLM + citation prompt, intent router, Streamlit UI, RAGAS eval.